# 11. End-to-End RAG Pipeline with a Gradio UI

**RAG Pipeline Series — Notebook 11**

Notebooks 1-10 built and measured every piece of the *retrieval* half of RAG: loading and chunking `rag.pdf` (1-2), sparse and dense representations (3-4), vector storage and metadata filtering (5-6), retriever interfaces and hybrid fusion (7-8), measurement (9), and re-ranking (10). `rag.pdf`'s own Chapter 9 (Augmentation) and Chapter 10 (Generation) pick up where retrieval leaves off: once you have the right chunks, you still have to *assemble* them into a prompt (augmentation) and hand that prompt to an LLM to produce a grounded answer (generation).

This notebook closes the loop end-to-end:

1. Rebuild the dense retriever from earlier notebooks (load → chunk → embed → index).
2. **Augment**: stuff the retrieved chunks into a prompt template alongside the user's question.
3. **Generate**: send that prompt to Google's Gemini via `langchain-google-genai`, using a free-tier model.
4. Wrap steps 1-3 in a single `rag_answer()` function, then expose it through a small **Gradio** chat-style UI — type a question, get a grounded answer back, sources and all.

The notebook is written to run standalone in Google Colab as well as locally.

## Setup

In [5]:
%pip install -q -U langchain_google_genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -q -U langchain langchain-community langchain-core langchain-text-splitters sentence-transformers langchain-huggingface langchain-chroma chromadb langchain-google-genai python-dotenv gradio pandas

### Files this notebook needs

- `rag.pdf` — the source document (same as every other notebook in the series).
- `.env` — must contain a `GOOGLE_API_KEY` for Gemini. Get a free key from [Google AI Studio](https://aistudio.google.com/apikey); the `gemini-2.5-flash` model used below is available on the free tier.

In Colab neither file exists on the VM yet, so the cell below opens a file picker twice — once for `rag.pdf`, once for `.env` — via the same `maybe_colab_upload()` helper the series already uses. Locally, both files already sit next to this notebook, so the picker is skipped.

In [ ]:
from rag_utils import maybe_colab_upload

# Only runs inside Colab. Opens a file picker each time; select rag.pdf, then .env.
# Safe to skip this cell if you're running locally and already have both files on disk.
maybe_colab_upload()  # -> rag.pdf
maybe_colab_upload()  # -> .env

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Colab uploads land in /content; locally the .env sits next to this notebook.
env_path = "/content/.env" if os.path.exists("/content/.env") else ".env"
assert os.path.exists(env_path), "Could not find .env - upload it first (see cell above)."
load_dotenv(env_path)

assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set - check your .env file."
print("GOOGLE_API_KEY loaded.")

GOOGLE_API_KEY loaded.


## 1. Rebuild the retriever

Same pipeline as notebooks 7-10: load `rag.pdf`, strip headers/footers, chunk chapter-by-chapter, embed with the series' `sentence-transformers/paraphrase-MiniLM-L3-v2` model, and index into an in-memory Chroma store. `k=5` matches the default used before notebook 10 widened the net for re-ranking — this notebook isn't re-ranking, so there's no need for a wider shortlist.

In [2]:
from rag_utils import build_chroma_store, get_embedder, load_chapter_chunks

pages, full_text, chapters, chunks = load_chapter_chunks()

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(f"{len(chunks)} chunks indexed; retriever returns top 5 matches per query")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7860.93it/s]


181 chunks indexed; retriever returns top 5 matches per query


## 2. Augmentation: turning retrieved chunks into a prompt

Chapter 9's framing of augmentation is simple: the retrieved chunks aren't the answer, they're *context* — text inserted into a prompt so the model answers from the document instead of its own parametric memory. `format_context()` labels each chunk with the chapter it came from (useful both for the model's citations and for our own debugging), and `RAG_PROMPT` is a system/human template that instructs the model to stick to that context.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant answering questions about a RAG course document. "
     "Answer using ONLY the context below - do not use outside knowledge. "
     "If the answer isn't in the context, say you don't know. "
     "Mention which chapter(s) your answer came from."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])


def format_context(docs):
    return "\n\n".join(
        f"[Chapter {d.metadata['chapter_num']}: {d.metadata['chapter_title']}]\n{d.page_content}"
        for d in docs
    )

## 3. Generation: Gemini via `langchain-google-genai`

`ChatGoogleGenerativeAI` picks up `GOOGLE_API_KEY` from the environment automatically, so no key needs to be passed in code. `gemini-2.5-flash` is Google's fast, free-tier-eligible model — a good fit for a small RAG demo like this one (Chapter 10 makes the same generation-model tradeoff Chapter 8 made for re-rankers: a bigger model is more accurate but slower and costlier, so start with the cheapest model that's good enough).

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.2)

## 4. Wiring it together: `rag_answer()`

One function, three steps: retrieve, augment, generate. This is the entire RAG pipeline the series has been building toward - everything from notebooks 1-10 lives inside `retriever`, and steps 2-3 above turn its output into a grounded answer.

In [8]:
def rag_answer(question, k=5):
    docs = retriever.invoke(question)[:k]
    context = format_context(docs)
    messages = RAG_PROMPT.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    return response.content, docs

Try it on one question before wiring up a UI, so any problem (missing key, empty retrieval, model error) shows up here first.

In [9]:
answer, docs = rag_answer("What is re-ranking used for in a RAG pipeline?")

print("ANSWER:\n", answer)
print("\nSOURCES:")
for d in docs:
    print(f"  - Chapter {d.metadata['chapter_num']}: {d.metadata['chapter_title']}")

ANSWER:
 [{'type': 'text', 'text': 'In a RAG pipeline, re-ranking is used as a precision-focused second stage in a multi-stage retrieval funnel. Its purpose is to:\n\n*   **Surgically select the most relevant documents:** It takes a larger pool of candidates (top-100 or top-500) from the first stage and narrows them down to the top-5 or top-10.\n*   **Increase precision:** While the first stage focuses on high recall (casting a wide net), the re-ranking stage uses a powerful cross-encoder model to ensure high precision.\n*   **Manage latency:** This approach allows the system to achieve high-quality results within an acceptable total latency budget (typically < 100ms).\n\n**Source:** Chapter 08: Re-ranking', 'extras': {'signature': 'Eq0LCqoLARFNMg8mqGueFgh+XYltDdpVqOaR1szyKkJSOj+ZWvjBkFoHyI61crmj4Arl395ddV4/eibfeXfb/g2cF6k7HoacGK4jx8E3nUNkpL47wEDnIihPVrCb2fLqZHUQYdF62ftSFs/vAClU6rCs9zwzZgVYrZfq8RnLfd93NLRzoqLovkva7tpO2C4MrQTzMA+y1Dbwrx9aFEDb1k5kUU4LW0dIE/cuXJ5S930r1ec5EDnMbamqq2xNza/jY

## 5. A Gradio UI

A minimal `gr.Blocks` app: a question box, an "Ask" button, and two output boxes — the generated answer, and the chapters it was grounded in. Clicking "Ask" (or submitting the textbox) runs the full retrieve → augment → generate pipeline and writes the result straight into the UI.

`demo.launch(share=True, debug=True)` is the Colab-safe way to launch: Colab can't serve a local port directly, so Gradio needs `share=True` to tunnel the UI to a public URL it prints below the cell. This works locally too - it just also opens the tunnel there.

In [11]:
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.


In [13]:
import gradio as gr


def ask(question):
    if not question or not question.strip():
        return "Please enter a question.", ""
    answer, docs = rag_answer(question)
    sources = "\n".join(
        f"- Chapter {d.metadata['chapter_num']}: {d.metadata['chapter_title']}" for d in docs
    )
    answer = answer[0]['text']
    return answer, sources


with gr.Blocks(title="RAG over rag.pdf") as demo:
    gr.Markdown(
        "# Ask rag.pdf\n"
        "End-to-end RAG: dense retrieval (notebooks 1-10) + Gemini generation (this notebook)."
    )
    question_box = gr.Textbox(label="Your question", placeholder="e.g. What is re-ranking used for?")
    ask_btn = gr.Button("Ask")
    answer_box = gr.Textbox(label="Answer", lines=6)
    sources_box = gr.Textbox(label="Retrieved chapters", lines=4)

    ask_btn.click(fn=ask, inputs=question_box, outputs=[answer_box, sources_box])
    question_box.submit(fn=ask, inputs=question_box, outputs=[answer_box, sources_box])

demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0936a67512a91de61f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0936a67512a91de61f.gradio.live


## Takeaways

- **Augmentation** is just prompt construction: label retrieved chunks with metadata (here, chapter number/title) and instruct the model to answer only from that context.
- **Generation** with `langchain-google-genai`'s `ChatGoogleGenerativeAI` needs nothing but `GOOGLE_API_KEY` in the environment and a model name — `gemini-2.5-flash` is fast and free-tier-eligible, a sensible default before reaching for a larger model.
- The entire series compresses into one function, `rag_answer()`: retrieve with the notebook 7-10 pipeline, augment with a prompt template, generate with an LLM.
- A UI doesn't need to be complicated: `gr.Blocks` wraps `rag_answer()` in a textbox-in, textbox-out app, and `share=True` is the one flag that makes it work in both Colab and locally.

This wraps up the RAG fundamentals series: ingestion and chunking, sparse/dense/hybrid retrieval, evaluation and re-ranking, and finally augmentation + generation behind a real UI - the same pipeline `rag.pdf`'s Chapters 1-10 walk through, applied to itself.